# 文章の多クラス分類
概要  
ライブドアニュースコーパスを用いてニュース記事から記事のラベルごとに分類を行う。使用するモデルはLSTM(単層か二層、ドロップアウト有無のスイッチ可)で、バッチサイズ、埋め込み層の次元、隠れ層の次元、学習率、ドロップアウトレシオのハイパパラメータを変化させてモデルの性能への影響を検証する。
  

In [ ]:
#課題の概要
#1.ipynbに学習結果・予測モデルが出力されている
#2.理したうえで独自の工夫がされているコメントがある
    #画像処理の場合
    #pythonによる画像データの水増し
    #GANを使用して画像生成
    #ハイパラメータの最適化
    #NG:根拠なく説明変数を増やす、タスクに足して不適切なデータ前処理
#3.結果についての考察がされている
    #学習結果を踏まえて考察する。
        #なぜlossが減らない -> なぜかを考える
    #今後の精度向上につながる内容を調査し記載する。
        #より発展させてやりたいモデル   最新論文などから最適化手法をpickup

#4.深層学習（中間層3層以上）を利用
    #ランダムフォレストなどの機械学習手法はNG

#開発のロードマップ
#テーマ設定 -> データ取集 -> アルゴリズム選定 -> モデル作成 -> 学習
#-> 評価 -> 考察 -> 提出

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import sys
sys.path.append('/content/gdrive/MyDrive/my_modules')

In [ ]:
#形態素解析に必要なライブラリの導入

!apt install aptitude swig
!aptitude install mecab libmecab-dev mecab-ipadic-utf8 git make curl xz-utils file -y
!pip install mecab-python3==0.996.3
!git clone --depth 1 https://github.com/neologd/mecab-ipadic-neologd.git
!echo yes | mecab-ipadic-neologd/bin/install-mecab-ipadic-neologd -n -a

#データの用意

In [ ]:
import pandas as pd
import MeCab
import subprocess
import re
import torch

In [ ]:
import pandas as pd
#ライブドアニュースコーパスのロード
datasets = pd.read_csv('/content/gdrive/MyDrive/AVILEN/e/product_ensyu/livedoor_news_corpus.csv',
                 usecols=['category', 'text'])
categories = set(datasets['category'])

In [ ]:
cmd = 'echo `mecab-config --dicdir`"/mecab-ipadic-neologd"'
path = (subprocess.Popen(cmd, stdout=subprocess.PIPE,
                           shell=True).communicate()[0]).decode('utf-8')
tagger = MeCab.Tagger("-Owakati -d {0}".format(path))

def make_wakati(sentence):
    # MeCabで分かち書き
    sentence = tagger.parse(sentence)
    # 半角全角英数字除去
    sentence = re.sub(r'[0-9０-９a-zA-Zａ-ｚＡ-Ｚ]+', " ", sentence)
    # 記号もろもろ除去
    sentence = re.sub(r'[\．_－―─！＠＃＄％＾＆\-‐|\\＊\“（）＿■×+α※÷⇒—●★☆〇◎◆▼◇△□(：〜～＋=)／*&^%$#@!~`){}［］…\[\]\"\'\”\’:;<>?＜＞〔〕〈〉？、。・,\./『』【】「」→←○《》≪≫\n\u3000]+', "", sentence)
    # スペースで区切って形態素の配列へ
    wakati = sentence.split(" ")
    # 空の要素は削除
    wakati = list(filter(("").__ne__, wakati))
    return wakati

In [ ]:
# 単語ID辞書を作成する
word2index = {}
word2index.update({"<pad>":0})

for text in datasets["text"]:
    wakati = make_wakati(text)
    for word in wakati:
        if word in word2index: continue
        word2index[word] = len(word2index)
print("vocab size : ", len(word2index))

In [ ]:
from sklearn.model_selection import train_test_split
import random
from sklearn.utils import shuffle
from sklearn.model_selection import KFold
from tqdm.notebook import tqdm
import numpy
import torch
import torch.utils.data

# ラベルID辞書を作成する
cat2index = {}
categories = set(datasets['category'])
for cat in categories:
    if cat in cat2index: continue
    cat2index[cat] = len(cat2index)

def sentence2index(sentence):
    wakati = make_wakati(sentence)
    return [word2index[w] for w in wakati]

def category2index(cat):
    return [cat2index[cat]]

index_datasets_text_tmp = []
index_datasets_category = []

# 系列の長さの最大値を取得。この長さに他の系列の長さをあわせる
max_len = 0
for text, category in zip(datasets["text"], datasets["category"]):
  index_text = sentence2index(text)
  index_category = category2index(category)
  index_datasets_text_tmp.append(index_text)
  index_datasets_category.append(index_category)
  if max_len < len(index_text):
    max_len = len(index_text)

# 系列の長さを揃えるために短い系列にパディングを追加
# 後ろパディングだと正しく学習できなかったので、前パディング
#文章データのidx化
index_datasets_text = []
for text in index_datasets_text_tmp:
  for i in range(max_len - len(text)):
    text.insert(0, 0) # 前パディング
  index_datasets_text.append(text)



In [ ]:
cd /content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment

In [ ]:
#上のコードでデータのid化したものをpickleを用いて保存したものをロードする。
import pickle

with open('/content/gdrive/MyDrive/AVILEN/e/product_ensyu/rakuten_data/word2index.pickle', 'rb') as f:
    word2index = pickle.load(f)
with open('/content/gdrive/MyDrive/AVILEN/e/product_ensyu/rakuten_data/cat2index.pickle', 'rb') as f:
    cat2index = pickle.load(f)
with open('/content/gdrive/MyDrive/AVILEN/e/product_ensyu/rakuten_data/index_datasets_category.pickle', 'rb') as f:
    index_datasets_category = pickle.load(f)
with open('/content/gdrive/MyDrive/AVILEN/e/product_ensyu/rakuten_data/index_datasets_text.pickle', 'rb') as f:
    index_datasets_text = pickle.load(f)

In [ ]:
#データセットをラッピングするためのクラスの宣言
import torch

class BuildDataset(torch.utils.data.Dataset):
    def __init__(self, text, label, transform_text=None, transform_label=None):
        self.text = text
        self.label = label
        self.transform_text = transform_text
        self.transform_label = transform_label
    def __len__(self):
        return len(self.text)
    def __getitem__(self, idx):
        if self.transform_text:
            self.text = self.transform_text(text)
        if self.transform_label:
            self.label = self.transform_label(label)
        text_ = torch.LongTensor(self.text[idx])
        label_ = torch.LongTensor(self.label[idx])
        return text_, label_

#サブセットクラスの宣言
class Subset2(torch.utils.data.Subset):
    def __init__(self, dataset, indices):
        super().__init__(dataset, indices)
        self.text = dataset.text
        self.label = dataset.label
        self.dataset = dataset
        self.indices = indices

In [ ]:
#全体のデータセットをpytorchで解析するためにラッピング
dataset = BuildDataset(index_datasets_text, index_datasets_category)

In [ ]:
#データ分割
from sklearn.model_selection import train_test_split
test_size = 0.3
val_size = 0.3
#trainva(訓練用と検証用の両方)lとtest(テスト)に用いるデータのそれぞれのインデックスを抽出
trainval_idx, test_idx  = train_test_split(list(range(len(dataset.label))), test_size=test_size, random_state=2021, shuffle=True)
#指定のインデックスからtrainvalのデータセットを作成
trainval_dataset = Subset2(dataset, trainval_idx)
#trainvalからtrainとvalidationのインデックスを抽出
train_idx, val_idx = train_test_split(list(range(len(trainval_dataset))), test_size=val_size, random_state=2021, shuffle=True)
#train、validation、test用のそれぞれのデータセットを作成
train_dataset = Subset2(trainval_dataset, train_idx)
val_dataset = Subset2(trainval_dataset, val_idx)
test_dataset = Subset2(dataset, test_idx)

# 必要なクラスや関数の宣言

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type== 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

### 使用するLSTM分類器モデルのクラス  
単層か2層のLSTM、ドロップアウトをスイッチできる。

In [ ]:
# 単層か2層のLSTM、ドロップアウトをスイッチできるクラス

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
#GPUで並列化処理による高速化。
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#LSTM分類器クラスの宣言
#ハイパパラメータは辞書で宣言
class LSTMClassifier(nn.Module):
    #各層の宣言　6層
    def __init__(self, **params):
        super(LSTMClassifier, self).__init__()
        #埋め込み層の次元の宣言
        self.word_embeddings = nn.Embedding(params["vocab_size"], params["embedding_dim"], padding_idx=0)
        #dropout1のインスタンス化
        self.dropout1 = nn.Dropout(params["dropout_ratio"])
        #lstm1のインスタンス化
        self.lstm1 = nn.LSTM(params["embedding_dim"], params["hidden_dim"], batch_first=True)
        #dropout2のインスタンス化
        self.dropout2 = nn.Dropout(params["dropout_ratio"])
        #lstm2のインスタンス化
        self.lstm2 = nn.LSTM(params["hidden_dim"], params["hidden_dim"], batch_first=True)
        #dropout3のインスタンス化
        self.dropout3 = nn.Dropout(params["dropout_ratio"])
        #lstmの出力を全結合しtag(ラベル)を出力
        self.linear = nn.Linear(params["hidden_dim"], params["tag_size"])
        #softmaxでラベルが取る確率を算出する。
        self.softmax = nn.LogSoftmax(dim=1)

    #各層を繋げて実行
    def forward(self, sentence):
        lstm1_out = None
        lstm2_out = None
        tag_scores = None
        embeds = self.word_embeddings(sentence)
        if params["dropout_flg"]:    #dropoutを使用するかのスイッチ
            emdeds = self.dropout1(embeds)
        if params["lstm_flg"]:   #2層のLSTMを使用をするかのスイッチ
            lstm1_out, _ = self.lstm1(embeds)
            if params["dropout_flg"]:
                lstm1_out = self.dropout2(lstm1_out)
            _, lstm2_out = self.lstm2(lstm1_out)
            lstm2_out = lstm2_out[0]
            if params["dropout_flg"]:
                lstm2_out = self.dropout3(lstm2_out)
            tag_space = self.linear(lstm2_out.view(-1, params["hidden_dim"]))
            tag_scores = self.softmax(tag_space)
        else:
            _, lstm1_out = self.lstm1(embeds)
            lstm1_out = lstm1_out[0]
            if params["dropout_flg"]:
                lstm1_out = self.dropout2(lstm1_out)
            tag_space = self.linear(lstm1_out.view(-1, params["hidden_dim"]))
            tag_scores = self.softmax(tag_space)

        return tag_scores

In [ ]:
#h1,h2なしに対応
#trainとvalidationの関数化
def train_val():
    _train_loss = 0.0
    _train_acc = 0.0
    model.train()
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        x_batch = x_batch.to(device).to(torch.int64) #データをGPUへ送る
        y_batch = y_batch.to(device).reshape(-1) #pytorchのモデルで訓練させるためにreshapeさせる
        optimizer.zero_grad() #バッチごとに勾配の初期化
        out = model(x_batch) #データをモデルに入力して出力
        loss = loss_function(out, y_batch) # lossを求める
        _, preds = torch.max(out, 1) # モデルの予測値を出力
        loss.backward() # バックプロパゲーションの計算
        optimizer.step() # パラメータの更新
        _train_loss += loss.item() #1バッチでのlossの累積を行う
        _train_acc += torch.sum(preds == y_batch).item() #1バッチでのデータのラベルと予測値が同じ個数なら累積させる
    train_loss.append(_train_loss) #各バッチごとの訓練誤差を記録
    _train_epoch_acc = _train_acc / len(train_loader.dataset) #全バッチを通しての訓練時の正解率を算出
    train_acc.append(_train_epoch_acc) #バッチごとに正解率を記録
    print("Epoch: {}, train_loss: {}, train_acc: {}%".format(
           epoch + 1, round(_train_loss, 2), round(100 * _train_epoch_acc, 2)))
    print("Epoch: {}, train_loss: {}, train_acc: {}%".format(
           epoch + 1, round(_train_loss, 2), round(100 * _train_epoch_acc, 2)), file=f)


    _val_loss = 0.0
    _val_acc = 0.0
    model.eval() #訓練後のモデルの検証
    with torch.no_grad(): #学習は行わないので勾配の計算はoffにする。
        for x_batch, y_batch in tqdm(val_loader):
            x_batch = x_batch.to(device).to(torch.int64)
            y_batch = y_batch.to(device).reshape(-1)
            optimizer.zero_grad()
            out = model(x_batch)
            loss = loss_function(out, y_batch)
            _, preds = torch.max(out, 1)
            _val_loss += loss.item()
            _val_acc += torch.sum(preds == y_batch).item()
        val_loss.append(_val_loss)
        _val_epoch_acc = _val_acc / len(val_loader.dataset)
        val_acc.append(_val_epoch_acc)
        train_val_evals["epoch_"+str(epoch+1)] = {"train_loss":_train_loss, "train_acc":_train_epoch_acc, "val_loss":_val_loss, "val_acc":_val_epoch_acc}
        print("Epoch: {}, val_loss: {}, val_acc: {}%\n".format(
            epoch + 1, round(_val_loss, 2), round(100 * _val_epoch_acc, 2)))
        print("Epoch: {}, val_loss: {}, val_acc: {}%\n".format(
            epoch + 1, round(_val_loss, 2), round(100 * _val_epoch_acc, 2)),file=f)

#テストの関数化
def test():
    _test_loss = 0.0
    _test_acc = 0.0
    model.eval()
    with torch.no_grad():
        for x_batch, y_batch in tqdm(test_loader):
            x_batch = x_batch.to(device).to(torch.int64)
            y_batch = y_batch.to(device).reshape(-1)
            out = model(x_batch)
            loss = loss_function(out, y_batch)
            _, y_pred = torch.max(out, 1)
            _test_loss += loss.item()
            _test_acc += torch.sum(y_pred == y_batch).item()
            y_batch = y_batch.cpu()
            y_pred = y_pred.cpu() 
            y_batch_total.extend(y_batch.tolist())
            y_pred_total.extend(y_pred.tolist()) 
        test_loss.append(_test_loss)
        _test_epoch_acc = _test_acc / len(test_loader.dataset)
        test_acc.append(_test_epoch_acc)
        global y_pred_test
        y_pred_test = {"y_pred":y_pred_total, "y_test":y_batch_total}
        global test_evals
        test_evals = {"test_loss":_test_loss, "test_acc":_test_epoch_acc}
        print("test_loss: {}, test_acc: {}%\n".format(round(_test_loss, 2), round(100 * _test_epoch_acc, 2)))
        print("test_loss: {}, test_acc: {}%\n".format(round(_test_loss, 2), round(100 * _test_epoch_acc, 2)),file=f)
        #多クラスのクロス集計表
        df_accuracy = pd.DataFrame({'y_pred': y_pred_total,
                        'y_test': y_batch_total})
        print("多クラスの混同行列")
        print("多クラスの混同行列", file=f)
        print(pd.crosstab(df_accuracy['y_pred'], df_accuracy['y_test']))
        print(pd.crosstab(df_accuracy['y_pred'], df_accuracy['y_test']),file=f)


In [ ]:
# import matplotlib.pyplot as plt
# #train_lossとval_lossをグラフで可視化
# #train_accとval_accをグラフで可視化
# #train_loss vs val_loss, train_acc vs val_acc, train_loss vs test_loss, train_acc vs test_acc
# #args = {train_loss: data1, val_loss: data2, loss_or_acc: "loss"}

# class SetGragh():
#     def __init__(self):
#         self.fig = plt.figure()
#         self.ax = self.fig1.add_subplot(111)

#     def run(self, loss_or_acc, **args):
#         prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_figs/"
        
#         fullpath = prepath + filename
#         for key, value in args.items():
#             if key == "loss_or_acc":
#                 if value == "loss":
#                     self.ax.set_ylabel('loss')
#                 else:
#                     self.ax.set_ylabel('acc')
#                 continue
#             self.ax.plot(range(1, epoch+2), value, label=key)
#         self.ax.set_xlabel('epoch')
#         self.ax.legend(loc='best')
#         if save_flg == 1:
#             self.fig1.savefig(fullpath)

## グラフ

In [ ]:
import matplotlib.pyplot as plt
#train_lossとval_lossをグラフで可視化
#train_accとval_accをグラフで可視化
class SetGragh():
    def __init__(self):
        self.fig1 = plt.figure()
        self.fig2 = plt.figure()
        self.ax1 = self.fig1.add_subplot(111)
        self.ax2 = self.fig2.add_subplot(111)

    def run(self, train_loss, val_loss, train_acc, val_acc, filename1, filename2, save_flg):
        prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_figs/"
        
        fullpath1 = prepath + filename1
        fullpath2 = prepath + filename2 
        
        self.ax1.plot(range(1, epoch+2), train_loss, label='train_loss')
        self.ax1.plot(range(1, epoch+2), val_loss, label='val_loss')
        self.ax1.set_xticks(np.arange(1, epoch+2, 1))
        self.ax1.set_ylabel('loss')
        self.ax1.set_xlabel('epoch')
        self.ax1.legend(loc='best')
        if save_flg == 1:
            self.fig1.savefig(fullpath1)

        self.ax2.plot(range(1, epoch+2), train_acc, label='train_acc')
        self.ax2.plot(range(1, epoch+2), val_acc, label='val_acc')
        self.ax2.set_xticks(np.arange(1, epoch+2, 1))
        self.ax2.set_ylabel('acc')
        self.ax2.set_xlabel('epoch')
        self.ax2.legend(loc='best')
        if save_flg == 1:
            self.fig2.savefig(fullpath2)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
#train_lossとtrain_accをグラフで可視化
class SetGragh2():
    def run(self, epoch, train_loss, train_acc, filename, save_flg):
        prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_figs/"
        fullpath = prepath + filename

        fig = plt.figure()
        ax1 = fig.add_subplot(111)
        ax1.set_xticks(np.arange(1, epoch+2, 1))
        ln1=ax1.plot([i for i in range(1, epoch+1)], train_loss,'C0',label=r'$train loss$')

        ax2 = ax1.twinx()
        ax2.set_xticks(np.arange(1, epoch+2, 1))
        ln2=ax2.plot([i for i in range(1, epoch+1)], train_acc,'C1',label=r'$train acc$')

        h1, l1 = ax1.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax1.legend(h1+h2, l1+l2, loc='best')

        ax1.set_xlabel("epoch")
        ax1.set_ylabel("train loss")
        ax2.set_ylabel("train acc")
        if save_flg == 1:
            fig.savefig(fullpath)

In [ ]:
#マルチクラス分類の評価指標を計算
#accuracy, percison, recall, f1を求める。
#macro_percision, macro_recall, macro_f1は平均値。
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
import numpy as np
class MultiClassEval():
    def __init__(self, y_pred_test, filename, save_flg):
        self.y_pred_test = y_pred_test
        self.filename = filename
        self.save_flg = save_flg

    def run(self):
        #出力をファイルに保存する機能を付ける。
        prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_figs/"
        fullpath = prepath + self.filename
        with open(fullpath, "w") as f:
            pred_key, test_key = list(self.y_pred_test.keys())
            y_pred = self.y_pred_test[pred_key]
            y_test = self.y_pred_test[test_key]
            #多クラスの混同行列
            print("多クラスの混同行列\n")
            print("多クラスの混同行列\n", file=f)
            df_accuracy = pd.DataFrame({'y_pred': y_pred,
                                        'y_test': y_test})
            print(pd.crosstab(df_accuracy['y_pred'], df_accuracy['y_test']))
            print(pd.crosstab(df_accuracy['y_pred'], df_accuracy['y_test']), file=f)
            acc_score = accuracy_score(y_test, y_pred)
            print("\n\naccuracy : {}%\n".format(round(100*acc_score, 2)))
            print("\n\naccuracy : {}%\n".format(round(100*acc_score, 2)), file=f)
            pre_score = precision_score(y_test,y_pred,average=None)
            pre_score_r = [round(pre_score[n], 2) for n in range(len(pre_score))]
            print("\nprecision : {}\n".format(pre_score_r))
            print("\nprecision : {}\n".format(pre_score_r), file=f)
            print("macro_ave_precision : {}\n".format(round(np.array(pre_score_r).mean(), 2)))
            print("macro_ave_precision : {}\n".format(round(np.array(pre_score_r).mean(), 2)), file=f)
            recall = recall_score(y_test, y_pred, average=None)
            print("\nrecall : {}\n".format([round(recall[n], 2) for n in range(len(recall))]))
            print("\nrecall : {}\n".format([round(recall[n], 2) for n in range(len(recall))]), file=f)
            print("macro_recall : {}\n".format(round(np.array(recall).mean(), 2)))
            print("macro_recall : {}\n".format(round(np.array(recall).mean(), 2)), file=f)
            f1 = f1_score(y_test, y_pred, average=None)
            print("\nf1 : {}\n".format([round(f1[n], 2) for n in range(len(f1))]))
            print("\nf1 : {}\n".format([round(f1[n], 2) for n in range(len(f1))]), file=f)
            print("macro_f1 : {}\n".format(round(np.array(f1).mean(), 2)))
            print("macro_f1 : {}\n".format(round(np.array(f1).mean(), 2)), file=f)

# 使用できるGPUの確認


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.memory_reserved(device=device))
print(torch.cuda.memory_allocated(device=device))

# 検証  
モデルの概要## ()は選択  

input_data => (dropout) => emdedding_layer => (dropout) =>    lstm_layer => (dropout) => (lstm_layer) => linear_layer => softmax_layer => output


# Dropoutを用いた検証
Dropoutを導入するかで単層のLSTMモデルの性能にどう影響が出るのか検証する。

## Dropoutなしのモデル

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": False, #単層か2層のLSTMのスイッチ
    "dropout_flg": False ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/single_lstm/"  #保存先のパス
filename = "without_dropout.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
without_dropout_evals_trainval = train_val_evals
without_dropout_evals_test = test_evals
without_dropout_pred_test = y_pred_test

without_dropout_ta = []
without_dropout_tl = []
without_dropout_va = []
without_dropout_vl = []
for key in without_dropout_evals_trainval.keys():
    without_dropout_ta.append(without_dropout_evals_trainval[key]["train_acc"])
    without_dropout_tl.append(without_dropout_evals_trainval[key]["train_loss"])
    without_dropout_va.append(without_dropout_evals_trainval[key]["val_acc"])
    without_dropout_vl.append(without_dropout_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "single_lstm/without_dropout_1.png" 
filename2 = "single_lstm/without_dropout_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(without_dropout_tl, without_dropout_vl, without_dropout_ta, without_dropout_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "single_lstm/without_dropout_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], without_dropout_tl, without_dropout_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = without_dropout_pred_test
filename = "single_lstm/without_dropout_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_without_dropout = []
evals_y_without_dropout.append(without_dropout_ta)
evals_y_without_dropout.append(without_dropout_tl)
evals_y_without_dropout.append(without_dropout_va)
evals_y_without_dropout.append(without_dropout_vl)
evals_y_without_dropout.append(without_dropout_vl)
evals_y_without_dropout.append(without_dropout_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "single_lstm/evals_y_without_dropout.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_without_dropout, f) #保存

## Dropoutありのモデル

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": False, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/single_lstm/"  #保存先のパス
filename = "with_dropout.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
with_dropout_evals_trainval = train_val_evals
with_dropout_evals_test = test_evals
with_dropout_pred_test = y_pred_test

with_dropout_ta = []
with_dropout_tl = []
with_dropout_va = []
with_dropout_vl = []
for key in with_dropout_evals_trainval.keys():
    with_dropout_ta.append(with_dropout_evals_trainval[key]["train_acc"])
    with_dropout_tl.append(with_dropout_evals_trainval[key]["train_loss"])
    with_dropout_va.append(with_dropout_evals_trainval[key]["val_acc"])
    with_dropout_vl.append(with_dropout_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "single_lstm/with_dropout_1.png" 
filename2 = "single_lstm/with_dropout_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(with_dropout_tl, with_dropout_vl, with_dropout_ta, with_dropout_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "single_lstm/with_dropout_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], with_dropout_tl, with_dropout_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = with_dropout_pred_test
filename = "single_lstm/with_dropout_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_with_dropout = []
evals_y_with_dropout.append(with_dropout_ta)
evals_y_with_dropout.append(with_dropout_tl)
evals_y_with_dropout.append(with_dropout_va)
evals_y_with_dropout.append(with_dropout_vl)
evals_y_with_dropout.append(with_dropout_vl)
evals_y_with_dropout.append(with_dropout_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "single_lstm/evals_y_with_dropout.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_with_dropout, f) #保存

## 考察
20epochでのDropoutなしのモデルの訓練誤差は16.36、検証誤差は16.43、汎化誤差は23.76、テストの正解率は69.88%になった。Dropoutありのモデルの訓練誤差は19.14、検証誤差は14.12、汎化誤差は19.31、テストの正解率は73.9%になった。Dropoutありの場合の方が訓練誤差よりも検証誤差、汎化誤差が大きくなることはなくテストの正解率も良くなっていることからDropoutは汎化性能を高めモデルの性能に寄与していることが分かる。Dropoutによりマスクされるノードをランダムに選択し学習の段階ごとに行うことでアンサンブル効果を得ていると考えられる。Dropoutを加えたモデルの方がモデルの性能に良い影響を与えやすいため以降はDropoutを使用したモデルを用いて実験を続ける。

# 複層のLSTM
単層か2層のLSTMを用いたそれぞれのモデルを比較してモデルに与える影響を考察する。

## 二層のLSTMのモデル

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": False ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "multi_lstm_without_dropout.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
multi_lstm_without_dropout_evals_trainval = train_val_evals
multi_lstm_without_dropout_evals_test = test_evals
multi_lstm_without_dropout_pred_test = y_pred_test

multi_lstm_without_dropout_ta = []
multi_lstm_without_dropout_tl = []
multi_lstm_without_dropout_va = []
multi_lstm_without_dropout_vl = []
for key in multi_lstm_without_dropout_evals_trainval.keys():
    multi_lstm_without_dropout_ta.append(multi_lstm_without_dropout_evals_trainval[key]["train_acc"])
    multi_lstm_without_dropout_tl.append(multi_lstm_without_dropout_evals_trainval[key]["train_loss"])
    multi_lstm_without_dropout_va.append(multi_lstm_without_dropout_evals_trainval[key]["val_acc"])
    multi_lstm_without_dropout_vl.append(multi_lstm_without_dropout_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/multi_lstm_without_dropout_1.png" 
filename2 = "multi_lstm/multi_lstm_without_dropout_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(multi_lstm_without_dropout_tl, multi_lstm_without_dropout_vl, multi_lstm_without_dropout_ta, multi_lstm_without_dropout_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/multi_lstm_without_dropout_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], multi_lstm_without_dropout_tl, multi_lstm_without_dropout_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = multi_lstm_without_dropout_pred_test
filename = "multi_lstm/multi_lstm_without_dropout_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_multi_lstm_without_dropout = []
evals_y_multi_lstm_without_dropout.append(multi_lstm_without_dropout_ta)
evals_y_multi_lstm_without_dropout.append(multi_lstm_without_dropout_tl)
evals_y_multi_lstm_without_dropout.append(multi_lstm_without_dropout_va)
evals_y_multi_lstm_without_dropout.append(multi_lstm_without_dropout_vl)
evals_y_multi_lstm_without_dropout.append(multi_lstm_without_dropout_vl)
evals_y_multi_lstm_without_dropout.append(multi_lstm_without_dropout_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_multi_lstm_without_dropout.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_multi_lstm_without_dropout, f) #保存

## 二層のLSTMモデル + Dropout

In [ ]:
# 20epochでのDropoutなしのモデルの訓練誤差は16.36、検証誤差は16.43、汎化誤差は23.76、テストの正解率は69.88%になった。

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "multi_lstm_with_dropout.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
multi_lstm_with_dropout_evals_trainval = train_val_evals
multi_lstm_with_dropout_evals_test = test_evals
multi_lstm_with_dropout_pred_test = y_pred_test

multi_lstm_with_dropout_ta = []
multi_lstm_with_dropout_tl = []
multi_lstm_with_dropout_va = []
multi_lstm_with_dropout_vl = []
for key in multi_lstm_with_dropout_evals_trainval.keys():
    multi_lstm_with_dropout_ta.append(multi_lstm_with_dropout_evals_trainval[key]["train_acc"])
    multi_lstm_with_dropout_tl.append(multi_lstm_with_dropout_evals_trainval[key]["train_loss"])
    multi_lstm_with_dropout_va.append(multi_lstm_with_dropout_evals_trainval[key]["val_acc"])
    multi_lstm_with_dropout_vl.append(multi_lstm_with_dropout_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/multi_lstm_with_dropout_1.png" 
filename2 = "multi_lstm/multi_lstm_with_dropout_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(multi_lstm_with_dropout_tl, multi_lstm_with_dropout_vl, multi_lstm_with_dropout_ta, multi_lstm_with_dropout_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/multi_lstm_with_dropout_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], multi_lstm_with_dropout_tl, multi_lstm_with_dropout_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = multi_lstm_with_dropout_pred_test
filename = "multi_lstm/multi_lstm_with_dropout_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_multi_lstm_with_dropout = []
evals_y_multi_lstm_with_dropout.append(multi_lstm_with_dropout_ta)
evals_y_multi_lstm_with_dropout.append(multi_lstm_with_dropout_tl)
evals_y_multi_lstm_with_dropout.append(multi_lstm_with_dropout_va)
evals_y_multi_lstm_with_dropout.append(multi_lstm_with_dropout_vl)
evals_y_multi_lstm_with_dropout.append(multi_lstm_with_dropout_vl)
evals_y_multi_lstm_with_dropout.append(multi_lstm_with_dropout_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_multi_lstm_with_dropout.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_multi_lstm_with_dropout, f) #保存

## 考察
20epochでの二層のLSTMを用いたモデルの訓練誤差は12.08、検証誤差は16.08、汎化誤差は22.1、正解率は71.82%であった。更にDropoutを加えた場合では、訓練誤差は14.1、検証誤差は14.62、汎化誤差は19.95、正解率は76.12%であった。(Dropoutなしの単層LSTMのモデルの訓練誤差は16.36、検証誤差は16.43、汎化誤差は23.76、テストの正解率は69.88%)
二層のLSTMのモデルの方が単層のLSTMの場合と比べて、訓練誤差が小さくなり検証誤差、汎化誤差を下回っていることから過学習していると分かる。二層にしたことで特徴量が増えたことでモデルが複雑になり過学習になったと考えられる。Dropoutを加えると、訓練誤差と検証誤差、汎化誤差との差分が狭まり過学習を抑制する効果があり、テストの正解率も向上した。二層のLSTMにDropoutを加えればモデルとして機能する可能性があると思われる。以降は実験的に二層のLSTMを用いて実験を続ける。

# その他のハイパラメータでの検証
比較する基準となるハイパパラメータの設定を2層のLSTM、ドロップアウトあり、ドロップアウトの割合を0.5、バッチサイズを100、学習率を0.001、埋め込み層の次元を20、隠れ層の次元を100とする。  
二層のLSTMモデル+Dropoutの実験結果より20epochの訓練誤差:14.1、検証誤差:14.62、汎化誤差:19.95、テストの正解率:76.12%をそれぞれの以下実験での比較対象とする。

## Batch_Size 50

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 50, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "bs_50.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
bs_50_evals_trainval = train_val_evals
bs_50_evals_test = test_evals
bs_50_pred_test = y_pred_test

bs_50_ta = []
bs_50_tl = []
bs_50_va = []
bs_50_vl = []
for key in bs_50_evals_trainval.keys():
    bs_50_ta.append(bs_50_evals_trainval[key]["train_acc"])
    bs_50_tl.append(bs_50_evals_trainval[key]["train_loss"])
    bs_50_va.append(bs_50_evals_trainval[key]["val_acc"])
    bs_50_vl.append(bs_50_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/bs_50_1.png" 
filename2 = "multi_lstm/bs_50_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(bs_50_tl, bs_50_vl, bs_50_ta, bs_50_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/bs_50_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], bs_50_tl, bs_50_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = bs_50_pred_test
filename = "multi_lstm/bs_50_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_bs_50 = []
evals_y_bs_50.append(bs_50_ta)
evals_y_bs_50.append(bs_50_tl)
evals_y_bs_50.append(bs_50_va)
evals_y_bs_50.append(bs_50_vl)
evals_y_bs_50.append(bs_50_vl)
evals_y_bs_50.append(bs_50_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_bs_50.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_bs_50, f) #保存

## Batch_Size 200

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 200, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "bs_200.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
bs_200_evals_trainval = train_val_evals
bs_200_evals_test = test_evals
bs_200_pred_test = y_pred_test

bs_200_ta = []
bs_200_tl = []
bs_200_va = []
bs_200_vl = []
for key in bs_200_evals_trainval.keys():
    bs_200_ta.append(bs_200_evals_trainval[key]["train_acc"])
    bs_200_tl.append(bs_200_evals_trainval[key]["train_loss"])
    bs_200_va.append(bs_200_evals_trainval[key]["val_acc"])
    bs_200_vl.append(bs_200_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/bs_200_1.png" 
filename2 = "multi_lstm/bs_200_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(bs_200_tl, bs_200_vl, bs_200_ta, bs_200_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/bs_200_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], bs_200_tl, bs_200_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = bs_200_pred_test
filename = "multi_lstm/bs_200_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_bs_200 = []
evals_y_bs_200.append(bs_200_ta)
evals_y_bs_200.append(bs_200_tl)
evals_y_bs_200.append(bs_200_va)
evals_y_bs_200.append(bs_200_vl)
evals_y_bs_200.append(bs_200_vl)
evals_y_bs_200.append(bs_200_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_bs_200.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_bs_200, f) #保存

## 考察
20epochでのバッチサイズ50のモデルの訓練誤差は19.85、検証誤差は27.81、汎化誤差は36.73、テストの正解率は76.93％、バッチサイズ200のモデルの訓練誤差は12.91、検証誤差は8.18、汎化誤差は12.06、テストの正解率は69.43％になった。（バッチサイズ100のモデルの訓練誤差は14.1、検証誤差は14.62、汎化誤差は19.95、正解率は76.12%）  
バッチサイズ50の結果よりバッチサイズを小さくすると汎化誤差は訓練誤差を大きく上回り過学習となりやすい。バッチサイズ100では若干過学習気味ではあるが訓練誤差と検証誤差はほぼ同じ値で汎化誤差は訓練誤差よりも5ほど高い程度でバッチサイズ50と比べて汎化性能は改善されている。テストの正解率は76.12%とバッチサイズ50の時と比べてもほとんど変わらない。さらにバッチサイズ200にすると正解率は69.43%と10％ほど落ちるが検証誤差、汎化誤差は訓練誤差を完全に下回りは汎化性能は向上する。バッチサイズを小さくするとエポックごとの学習でのパラメータを更新する回数が増えるので訓練データに適合されすぎると考えられる。  
後のハイパパラメータの再調整で汎化誤差、テストの正解率を見てバッチサイズを100以上で調整する。

## Emdedding dim 50

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 50, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "ed_50.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
ed_50_evals_trainval = train_val_evals
ed_50_evals_test = test_evals
ed_50_pred_test = y_pred_test

ed_50_ta = []
ed_50_tl = []
ed_50_va = []
ed_50_vl = []
for key in ed_50_evals_trainval.keys():
    ed_50_ta.append(ed_50_evals_trainval[key]["train_acc"])
    ed_50_tl.append(ed_50_evals_trainval[key]["train_loss"])
    ed_50_va.append(ed_50_evals_trainval[key]["val_acc"])
    ed_50_vl.append(ed_50_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/ed_50_1.png" 
filename2 = "multi_lstm/ed_50_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(ed_50_tl, ed_50_vl, ed_50_ta, ed_50_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/ed_50_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], ed_50_tl, ed_50_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = ed_50_pred_test
filename = "multi_lstm/ed_50_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_ed_50 = []
evals_y_ed_50.append(ed_50_ta)
evals_y_ed_50.append(ed_50_tl)
evals_y_ed_50.append(ed_50_va)
evals_y_ed_50.append(ed_50_vl)
evals_y_ed_50.append(ed_50_vl)
evals_y_ed_50.append(ed_50_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_ed_50.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_ed_50, f) #保存

##Embedding dim 100

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 100, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "ed_100.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
ed_100_evals_trainval = train_val_evals
ed_100_evals_test = test_evals
ed_100_pred_test = y_pred_test

ed_100_ta = []
ed_100_tl = []
ed_100_va = []
ed_100_vl = []
for key in ed_100_evals_trainval.keys():
    ed_100_ta.append(ed_100_evals_trainval[key]["train_acc"])
    ed_100_tl.append(ed_100_evals_trainval[key]["train_loss"])
    ed_100_va.append(ed_100_evals_trainval[key]["val_acc"])
    ed_100_vl.append(ed_100_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/ed_100_1.png" 
filename2 = "multi_lstm/ed_100_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(ed_100_tl, ed_100_vl, ed_100_ta, ed_100_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/ed_100_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], ed_100_tl, ed_100_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = ed_100_pred_test
filename = "multi_lstm/ed_100_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_ed_100 = []
evals_y_ed_100.append(ed_100_ta)
evals_y_ed_100.append(ed_100_tl)
evals_y_ed_100.append(ed_100_va)
evals_y_ed_100.append(ed_100_vl)
evals_y_ed_100.append(ed_100_vl)
evals_y_ed_100.append(ed_100_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_ed_100.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_ed_100, f) #保存

## 考察
20epochの埋め込み層の次元50のモデルの訓練誤差は12.38、検証誤差は16.03、汎化誤差は21.41、テストの正解率は73.13％、埋め込み層の次元100のモデルの訓練誤差は1.92、検証誤差は14.85、汎化誤差は19.92、テストの正解率は80.51％になった。（埋め込み層の次元20のモデルの訓練誤差は14.1、検証誤差は14.62、汎化誤差は19.95、正解率は76.12%）  
埋め込み層20次元の結果より検証誤差と汎化誤差が訓練誤差を下回っていることから埋め込み層の次元を小さくすると過学習を抑制していると分かる。一方で埋め込み次元を大きくすると検証誤差と汎化誤差が訓練誤差を上回り過学習になっている。単語のデータを大きな次元の埋め込み層に入力することで単語を表現する特徴量が増えるので学習する際に訓練データに適合され過ぎるためだと考えられる。テストの正解率はどのパターンでも70%以上を保ち、平均5%前後の変化ほどである。今回の実験では埋め込み次元を最大でも100次元ほどしか設定していなく埋め込み層はモデルの層の入力部でモデル全体からすると部分的にしか直接的に作用していないため次元を調整しても正解率に大きく影響しないと考えられる。  
後のハイパパラメータの再調整で汎化誤差やテストの正解率を見て埋め込み層の次元は50以下で調整する。

##Hidden dim 50

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 50, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "hd_50.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
hd_50_evals_trainval = train_val_evals
hd_50_evals_test = test_evals
hd_50_pred_test = y_pred_test

hd_50_ta = []
hd_50_tl = []
hd_50_va = []
hd_50_vl = []
for key in hd_50_evals_trainval.keys():
    hd_50_ta.append(hd_50_evals_trainval[key]["train_acc"])
    hd_50_tl.append(hd_50_evals_trainval[key]["train_loss"])
    hd_50_va.append(hd_50_evals_trainval[key]["val_acc"])
    hd_50_vl.append(hd_50_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/hd_50_1.png" 
filename2 = "multi_lstm/hd_50_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(hd_50_tl, hd_50_vl, hd_50_ta, hd_50_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/hd_50_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], hd_50_tl, hd_50_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = hd_50_pred_test
filename = "multi_lstm/hd_50_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_hd_50 = []
evals_y_hd_50.append(hd_50_ta)
evals_y_hd_50.append(hd_50_tl)
evals_y_hd_50.append(hd_50_va)
evals_y_hd_50.append(hd_50_vl)
evals_y_hd_50.append(hd_50_vl)
evals_y_hd_50.append(hd_50_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_hd_50.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_hd_50, f) #保存

In [ ]:
##モデルの概要## (dropout)は任意
"""
input_data => (dropout) => emdedding_layer => (dropout) => lstm_layer
=> (dropout) => lstm_layer => (dropout) => linear_layer => softmax_layer => outpot
"""

##Hidden dim 150

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 150, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "hd_150.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
hd_150_evals_trainval = train_val_evals
hd_150_evals_test = test_evals
hd_150_pred_test = y_pred_test

hd_150_ta = []
hd_150_tl = []
hd_150_va = []
hd_150_vl = []
for key in hd_150_evals_trainval.keys():
    hd_150_ta.append(hd_150_evals_trainval[key]["train_acc"])
    hd_150_tl.append(hd_150_evals_trainval[key]["train_loss"])
    hd_150_va.append(hd_150_evals_trainval[key]["val_acc"])
    hd_150_vl.append(hd_150_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/hd_150_1.png" 
filename2 = "multi_lstm/hd_150_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(hd_150_tl, hd_150_vl, hd_150_ta, hd_150_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/hd_150_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], hd_150_tl, hd_150_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = hd_150_pred_test
filename = "multi_lstm/hd_150_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_hd_150 = []
evals_y_hd_150.append(hd_150_ta)
evals_y_hd_150.append(hd_150_tl)
evals_y_hd_150.append(hd_150_va)
evals_y_hd_150.append(hd_150_vl)
evals_y_hd_150.append(hd_150_vl)
evals_y_hd_150.append(hd_150_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_hd_150.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_hd_150, f) #保存

##Hidden dim 300

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 300, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "hd_300.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
hd_300_evals_trainval = train_val_evals
hd_300_evals_test = test_evals
hd_300_pred_test = y_pred_test

hd_300_ta = []
hd_300_tl = []
hd_300_va = []
hd_300_vl = []
for key in hd_300_evals_trainval.keys():
    hd_300_ta.append(hd_300_evals_trainval[key]["train_acc"])
    hd_300_tl.append(hd_300_evals_trainval[key]["train_loss"])
    hd_300_va.append(hd_300_evals_trainval[key]["val_acc"])
    hd_300_vl.append(hd_300_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/hd_300_1.png" 
filename2 = "multi_lstm/hd_300_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(hd_300_tl, hd_300_vl, hd_300_ta, hd_300_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/hd_300_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], hd_300_tl, hd_300_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = hd_300_pred_test
filename = "multi_lstm/hd_300_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_hd_300 = []
evals_y_hd_300.append(hd_300_ta)
evals_y_hd_300.append(hd_300_tl)
evals_y_hd_300.append(hd_300_va)
evals_y_hd_300.append(hd_300_vl)
evals_y_hd_300.append(hd_300_vl)
evals_y_hd_300.append(hd_300_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_hd_300.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_hd_300, f) #保存

## 考察  
20epochの隠れ層の次元50のモデルの訓練誤差は27.45、検証誤差は16.61、汎化誤差は23.08、テストの正解率は66.62%、隠れ層の次元300のモデルの訓練誤差は14.81、検証誤差は15.9、汎化誤差は20.83、テストの正解率は73.9%になった。（隠れ層の次元100のモデルの訓練誤差は14.1、検証誤差は14.62、汎化誤差は19.95、正解率は76.12%）  
隠れ層次元50の結果より隠れ層の次元を小さくしたモデルの方が検証誤差と汎化誤差が訓練誤差を下回る結果となり過学習の抑制の効果があると分かる。隠れ層の次元を大きくしモデル全体のノード数が増えると訓練データに適合され過ぎるので隠れ層の次元は抑えた方がよいと考えられる。隠れ層の次元を小さくすると正解率は落ちるが300次元から20次元に極端に落としたとしてもテストの正解率は10%ほどの下落で抑えられている。ドロップアウトの効果もありテストの正解率に寄与しているためだと考えられる。  
後のハイパパラメータの再調整で汎化誤差、テストの正解率を見て隠れ層の次元は150以下で調整する。

##Lerning rate 0.0001

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-4, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "lr_04.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
lr_04_evals_trainval = train_val_evals
lr_04_evals_test = test_evals
lr_04_pred_test = y_pred_test

lr_04_ta = []
lr_04_tl = []
lr_04_va = []
lr_04_vl = []
for key in lr_04_evals_trainval.keys():
    lr_04_ta.append(lr_04_evals_trainval[key]["train_acc"])
    lr_04_tl.append(lr_04_evals_trainval[key]["train_loss"])
    lr_04_va.append(lr_04_evals_trainval[key]["val_acc"])
    lr_04_vl.append(lr_04_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/lr_04_1.png" 
filename2 = "multi_lstm/lr_04_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(lr_04_tl, lr_04_vl, lr_04_ta, lr_04_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/lr_04_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], lr_04_tl, lr_04_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = lr_04_pred_test
filename = "multi_lstm/lr_04_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_lr_04 = []
evals_y_lr_04.append(lr_04_ta)
evals_y_lr_04.append(lr_04_tl)
evals_y_lr_04.append(lr_04_va)
evals_y_lr_04.append(lr_04_vl)
evals_y_lr_04.append(lr_04_vl)
evals_y_lr_04.append(lr_04_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_lr_04.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_lr_04, f) #保存

##Lerning rate 0.01

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.5, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-2, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "lr_02.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
lr_02_evals_trainval = train_val_evals
lr_02_evals_test = test_evals
lr_02_pred_test = y_pred_test

lr_02_ta = []
lr_02_tl = []
lr_02_va = []
lr_02_vl = []
for key in lr_02_evals_trainval.keys():
    lr_02_ta.append(lr_02_evals_trainval[key]["train_acc"])
    lr_02_tl.append(lr_02_evals_trainval[key]["train_loss"])
    lr_02_va.append(lr_02_evals_trainval[key]["val_acc"])
    lr_02_vl.append(lr_02_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/lr_02_1.png" 
filename2 = "multi_lstm/lr_02_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(lr_02_tl, lr_02_vl, lr_02_ta, lr_02_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/lr_02_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], lr_02_tl, lr_02_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = lr_02_pred_test
filename = "multi_lstm/lr_02_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_lr_02 = []
evals_y_lr_02.append(lr_02_ta)
evals_y_lr_02.append(lr_02_tl)
evals_y_lr_02.append(lr_02_va)
evals_y_lr_02.append(lr_02_vl)
evals_y_lr_02.append(lr_02_vl)
evals_y_lr_02.append(lr_02_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_lr_02.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_lr_02, f) #保存

## 考察  
20epochの学習率0.01のモデルの訓練誤差は1.54、検証誤差は17.77、汎化誤差は23.76、テストの正解率は79.69%、学習率0.0001のモデルの訓練誤差は59.24、検証誤差は25.92、汎化誤差は37.33、テストの正解率は42.11%になった。（学習率0.001のモデルの訓練誤差は14.1、検証誤差は14.62、汎化誤差は19.95、正解率は76.12%）  
学習率0.01の結果より学習率を大きくすると訓練誤差は1.54に対して汎化誤差は23.76となり過学習に陥っていると分かる。学習率を上げるとパラメータを更新する速度が速くなり訓練誤差がすぐに収束してしまい、訓練データに過剰に適合するのが原因がと考えられる。学習率0.0001の結果より汎化性能を上げようと学習率を落とすと逆に学習が十分に行われなくテストの正解率を著しく下げてしまう。  
後のハイパパラメータの再調整で汎化誤差、テストの正解率を見て学習率は0.01未満で調整する。

##Dropout ratio 0.3

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.3, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "dr_03.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
dr_03_evals_trainval = train_val_evals
dr_03_evals_test = test_evals
dr_03_pred_test = y_pred_test

dr_03_ta = []
dr_03_tl = []
dr_03_va = []
dr_03_vl = []
for key in dr_03_evals_trainval.keys():
    dr_03_ta.append(dr_03_evals_trainval[key]["train_acc"])
    dr_03_tl.append(dr_03_evals_trainval[key]["train_loss"])
    dr_03_va.append(dr_03_evals_trainval[key]["val_acc"])
    dr_03_vl.append(dr_03_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/dr_03_1.png" 
filename2 = "multi_lstm/dr_03_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(dr_03_tl, dr_03_vl, dr_03_ta, dr_03_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/dr_03_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], dr_03_tl, dr_03_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = dr_03_pred_test
filename = "multi_lstm/dr_03_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_dr_03 = []
evals_y_dr_03.append(dr_03_ta)
evals_y_dr_03.append(dr_03_tl)
evals_y_dr_03.append(dr_03_va)
evals_y_dr_03.append(dr_03_vl)
evals_y_dr_03.append(dr_03_vl)
evals_y_dr_03.append(dr_03_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_dr_03.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_dr_03, f) #保存

##Dropout ratio 0.8

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.8, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 100, #バッチサイズ
    "lr": 1e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 100, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "dr_08.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
dr_08_evals_trainval = train_val_evals
dr_08_evals_test = test_evals
dr_08_pred_test = y_pred_test

dr_08_ta = []
dr_08_tl = []
dr_08_va = []
dr_08_vl = []
for key in dr_08_evals_trainval.keys():
    dr_08_ta.append(dr_08_evals_trainval[key]["train_acc"])
    dr_08_tl.append(dr_08_evals_trainval[key]["train_loss"])
    dr_08_va.append(dr_08_evals_trainval[key]["val_acc"])
    dr_08_vl.append(dr_08_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/dr_08_1.png" 
filename2 = "multi_lstm/dr_08_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(dr_08_tl, dr_08_vl, dr_08_ta, dr_08_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/dr_08_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], dr_08_tl, dr_08_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = dr_08_pred_test
filename = "multi_lstm/dr_08_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_dr_08 = []
evals_y_dr_08.append(dr_08_ta)
evals_y_dr_08.append(dr_08_tl)
evals_y_dr_08.append(dr_08_va)
evals_y_dr_08.append(dr_08_vl)
evals_y_dr_08.append(dr_08_vl)
evals_y_dr_08.append(dr_08_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_dr_08.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_dr_08, f) #保存

## 考察  
20epochのドロップアウトレシオ（ドロップアウトでノードをマスクする割合）0.3のモデルの訓練誤差は10.55、検証誤差は14.71、汎化誤差は20.76、テストの正解率は75.4%、ドロップアウトレシオ0.8のモデルの訓練誤差は38.18、検証誤差は16.54、汎化誤差は24.82、テストの正解率は69.52%になった。（ドロップアウトレシオ0.5のモデルの訓練誤差は14.1、検証誤差は14.62、汎化誤差は19.95、正解率は76.12%）  
ドロップアウトレシオ0.8の結果より検証誤差、汎化誤差が訓練誤差を下回っていることからドロップアウトレシオを大きくすると過学習を抑えられると分かる。ドロップアウトを大きくしても正解率は70%付近に留まり、大きく値を落とさないようである。学習ステップごとにランダムにノードをマスクし疑似的に複数の異なるモデルを再現し結果的にアンサンブル効果を得ていることから汎化性能に寄与していると考えられる。  
後のハイパパラメータの再調整で汎化誤差、テストの正解率を見てドロップアウトレシオは0.5以上で調整する。

# ハイパパラメータの再調整  
実験の結果を踏まえてハイパパラメータを再調整する。
- 二層のLSTMでドロップアウトあり  
- バッチサイズ：100以上
- 埋め込み層の次元：50以下
- 隠れ層の次元: 150以下
- 学習率： 0.01未満
- ドロップアウトレシオ：0.5以上

## テスト1  
バッチサイズ：150、埋め込み層の次元：20、隠れ層の次元:120、学習率：0.0008、ドロップアウトレシオ：0.6

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.6, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 150, #バッチサイズ
    "lr": 0.8e-3, #学習率
    "embedding_dim": 20, #埋め込み層の次元
    "hidden_dim": 120, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "adj_01.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
adj_01_evals_trainval = train_val_evals
adj_01_evals_test = test_evals
adj_01_pred_test = y_pred_test

adj_01_ta = []
adj_01_tl = []
adj_01_va = []
adj_01_vl = []
for key in adj_01_evals_trainval.keys():
    adj_01_ta.append(adj_01_evals_trainval[key]["train_acc"])
    adj_01_tl.append(adj_01_evals_trainval[key]["train_loss"])
    adj_01_va.append(adj_01_evals_trainval[key]["val_acc"])
    adj_01_vl.append(adj_01_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/adj_01_1.png" 
filename2 = "multi_lstm/adj_01_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(adj_01_tl, adj_01_vl, adj_01_ta, adj_01_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/adj_01_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], adj_01_tl, adj_01_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = adj_01_pred_test
filename = "multi_lstm/adj_01_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_adj_01 = []
evals_y_adj_01.append(adj_01_ta)
evals_y_adj_01.append(adj_01_tl)
evals_y_adj_01.append(adj_01_va)
evals_y_adj_01.append(adj_01_vl)
evals_y_adj_01.append(adj_01_vl)
evals_y_adj_01.append(adj_01_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_adj_01.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_adj_01, f) #保存

## テスト2  
バッチサイズ：150、埋め込み層の次元：15、隠れ層の次元:150、学習率：0.0008、ドロップアウトレシオ：0.6

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.6, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 150, #バッチサイズ
    "lr": 0.8e-3, #学習率
    "embedding_dim": 15, #埋め込み層の次元
    "hidden_dim": 150, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "adj_02.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
adj_02_evals_trainval = train_val_evals
adj_02_evals_test = test_evals
adj_02_pred_test = y_pred_test

adj_02_ta = []
adj_02_tl = []
adj_02_va = []
adj_02_vl = []
for key in adj_02_evals_trainval.keys():
    adj_02_ta.append(adj_02_evals_trainval[key]["train_acc"])
    adj_02_tl.append(adj_02_evals_trainval[key]["train_loss"])
    adj_02_va.append(adj_02_evals_trainval[key]["val_acc"])
    adj_02_vl.append(adj_02_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/adj_02_1.png" 
filename2 = "multi_lstm/adj_02_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(adj_02_tl, adj_02_vl, adj_02_ta, adj_02_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/adj_02_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], adj_02_tl, adj_02_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = adj_02_pred_test
filename = "multi_lstm/adj_02_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_adj_02 = []
evals_y_adj_02.append(adj_02_ta)
evals_y_adj_02.append(adj_02_tl)
evals_y_adj_02.append(adj_02_va)
evals_y_adj_02.append(adj_02_vl)
evals_y_adj_02.append(adj_02_vl)
evals_y_adj_02.append(adj_02_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_adj_02.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_adj_02, f) #保存

## テスト3  
バッチサイズ：200、埋め込み層の次元：18、隠れ層の次元:180、学習率：0.0008、ドロップアウトレシオ：0.6

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.6, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 200, #バッチサイズ
    "lr": 0.8e-3, #学習率
    "embedding_dim": 18, #埋め込み層の次元
    "hidden_dim": 180, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "adj_03.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
adj_03_evals_trainval = train_val_evals
adj_03_evals_test = test_evals
adj_03_pred_test = y_pred_test

adj_03_ta = []
adj_03_tl = []
adj_03_va = []
adj_03_vl = []
for key in adj_03_evals_trainval.keys():
    adj_03_ta.append(adj_03_evals_trainval[key]["train_acc"])
    adj_03_tl.append(adj_03_evals_trainval[key]["train_loss"])
    adj_03_va.append(adj_03_evals_trainval[key]["val_acc"])
    adj_03_vl.append(adj_03_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/adj_03_1.png" 
filename2 = "multi_lstm/adj_03_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(adj_03_tl, adj_03_vl, adj_03_ta, adj_03_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/adj_03_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], adj_03_tl, adj_03_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = adj_03_pred_test
filename = "multi_lstm/adj_03_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_adj_03 = []
evals_y_adj_03.append(adj_03_ta)
evals_y_adj_03.append(adj_03_tl)
evals_y_adj_03.append(adj_03_va)
evals_y_adj_03.append(adj_03_vl)
evals_y_adj_03.append(adj_03_vl)
evals_y_adj_03.append(adj_03_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_adj_03.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_adj_03, f) #保存

## テスト4
バッチサイズ：200、埋め込み層の次元：25、隠れ層の次元:180、
学習率：0.0008、ドロップアウトレシオ：0.6

In [ ]:
#ハイパパラメータの宣言
params = {
    "lstm_flg": True, #単層か2層のLSTMのスイッチ
    "dropout_flg": True ,#dropoutのスイッチ
    "dropout_ratio": 0.6, #dropoutの割合
    "epochs": 20, #エポック数
    "batch_size": 200, #バッチサイズ
    "lr": 0.8e-3, #学習率
    "embedding_dim": 25, #埋め込み層の次元
    "hidden_dim": 180, #隠れ層の次元
    "vocab_size": len(word2index), #全単語数
    "tag_size": len(categories), #ラベルの個数
    "test_size": test_size, #テスト用データに割り振る割合
    "val_size": val_size #検証用に割り振る割合
}

In [ ]:
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import time

start_time = time.time()
train_loss = []  #訓練誤差を格納
train_acc = []  #テストでの正解率を格納
val_loss = []  #検証誤差を格納
val_acc = []  #検証での正解率を格納
train_val_evals = {}  #訓練と検証の指標をまとめる
train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)  #DataLoaderのsamplerを用いてランダムにデータを抽出
val_subsampler = torch.utils.data.SubsetRandomSampler(val_idx)
train_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=train_subsampler, pin_memory=True,
                    num_workers=2)  #pin_memmoryとnum_workersを設定しモデル学習を高速化。
val_loader = torch.utils.data.DataLoader(
                    trainval_dataset, batch_size=params["batch_size"],
                    sampler=val_subsampler, pin_memory=True,
                    num_workers=2)
model = LSTMClassifier(**params).to(device)  #モデルのインスタンス化およびGPUへ送る
loss_function = nn.NLLLoss()  #損失関数
optimizer = optim.Adam(model.parameters(), lr=params["lr"])  #Adamによる最適化
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_results/multi_lstm/"  #保存先のパス
filename = "adj_04.txt"  #ファイル名
fullpath = prepath + filename
with open(fullpath, "w") as f:
    print("LSTMモデルの学習およびテスト結果\n")
    print("LSTMモデルの学習およびテスト結果\n", file=f)
    print("ハイパパラメータ\n{}\n".format(params))
    print("ハイパパラメータ\n{}\n".format(params), file=f)
    print("\n-------------------------学習-------------------------")
    print("\n-------------------------学習-------------------------", file=f)

#訓練と検証
for epoch in tqdm(range(params["epochs"])):
    with open(fullpath, "a") as f:
        print("-------------------------{}epoch-------------------------".format(epoch+1))
        print("-------------------------{}epoch-------------------------".format(epoch+1), file=f)
        train_val()
#テスト
test_evals = {} #指標を格納
y_pred_test = {} #予測値と真値の格納　
test_loader = torch.utils.data.DataLoader(
                      test_dataset, 
                      batch_size=params["batch_size"])
test_loss = []  #テスト誤差を格納
test_acc = []  #テストでの正解率を格納
y_batch_total = []  #真値を格納
y_pred_total = []  #モデルの出力値を格納
with open(fullpath, "a") as f:
    print("\n-------------------------テスト-------------------------".format(epoch+1))
    print("\n-------------------------テスト-------------------------".format(epoch+1), file=f)
    test()
    elapsed_time = time.time() - start_time #処理時間を計測
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]")
    print("\nelapsed_time:{}".format(round(elapsed_time, 2)) + "[sec]", file=f)
adj_04_evals_trainval = train_val_evals
adj_04_evals_test = test_evals
adj_04_pred_test = y_pred_test

adj_04_ta = []
adj_04_tl = []
adj_04_va = []
adj_04_vl = []
for key in adj_04_evals_trainval.keys():
    adj_04_ta.append(adj_04_evals_trainval[key]["train_acc"])
    adj_04_tl.append(adj_04_evals_trainval[key]["train_loss"])
    adj_04_va.append(adj_04_evals_trainval[key]["val_acc"])
    adj_04_vl.append(adj_04_evals_trainval[key]["val_loss"])

In [ ]:
#訓練と検証それぞれの損失を可視化
filename1 = "multi_lstm/adj_04_1.png" 
filename2 = "multi_lstm/adj_04_2.png"
save_flg = 1 # 保存は1とする。

SetGragh().run(adj_04_tl, adj_04_vl, adj_04_ta, adj_04_va, filename1, filename2, 1)

In [ ]:
#訓練の損失と正解率を可視化
filename = "multi_lstm/adj_04_3.png"
save_flg = 1
SetGragh2().run(params["epochs"], adj_04_tl, adj_04_ta, filename, save_flg)

In [ ]:
#テストの混同行列を表示
pred_test = adj_04_pred_test
filename = "multi_lstm/adj_04_4.txt"
save_flg = 1

MultiClassEval(pred_test, filename, save_flg).run()

In [ ]:
evals_y_adj_04 = []
evals_y_adj_04.append(adj_04_ta)
evals_y_adj_04.append(adj_04_tl)
evals_y_adj_04.append(adj_04_va)
evals_y_adj_04.append(adj_04_vl)
evals_y_adj_04.append(adj_04_vl)
evals_y_adj_04.append(adj_04_pred_test)

In [ ]:
import pickle
prepath = "/content/gdrive/My Drive/AVILEN/e/product_ensyu/Submission/submission_assignment/save_evals/"
filename = "multi_lstm/evals_y_adj_04.pkl"
fullpath = prepath + filename
with open(fullpath, "wb") as f:
    pickle.dump(evals_y_adj_04, f) #保存

## 結果
- テスト1(バッチサイズ：150、埋め込み層の次元：20、隠れ層の次元:120、学習率：0.0008、ドロップアウトレシオ：0.6)  
20epochでの訓練誤差は16.95、検証誤差は11.73、汎化誤差は15.32、テストの正解率は67.89%
- テスト2(バッチサイズ：150、埋め込み層の次元：15、隠れ層の次元:150、学習率：0.0008、ドロップアウトレシオ：0.6)  
20epochでの訓練誤差は16.05、検証誤差は10.83、汎化誤差は14.02、テストの正解率は70.06%  
- テスト3(バッチサイズ：200、埋め込み層の次元：18、隠れ層の次元:180、学習率：0.0008、ドロップアウトレシオ：0.6)  
20epochでの訓練誤差は15.6、検証誤差は8.22、汎化誤差は11.95、テストの正解率は68.66%
- テスト4(バッチサイズ：200、埋め込み層の次元：25、隠れ層の次元:180、 学習率：0.0008、ドロップアウトレシオ：0.6)  
20epochでの訓練誤差は11.89、検証誤差は6.83、汎化誤差は9.91、テストの正解率は74.99%  
  
バッチサイズを大きくし埋め込み層の次元、隠れ層の次元を合わせて大きくすると、訓練誤差よりも汎化誤差を小さくさせつつ、テストの正解率を75%台まで大きくすることが可能だと分かった。使用するGPUのメモリの制限があり学習することができなかったが、バッチサイズ300以上にして埋め込み層の次元、隠れ層の次元をより大きくすることで汎化性能を高めることが予測できる。

# 精度向上のための調査
コーパスのデータをより多く取集してデータ量によりモデルの精度を上げていく方法が考えられる。バッチデータはpytrochの機能より単純にランダムで選んだが、バッチごとにクラスの偏りがある可能性があり平準化することでモデルの学習向上に寄与すると考えられる。今回の実験では使用しなかったが重みの初期化なども考慮に加えたモデルを適用すればより詳細にデータを捉えることができると考えられる。以下に参考として載せる。
今回はLSTMのモデルで学習を行ったが、自然言語処理の多クラス分類タスクではその他にもCNNやself-attention,BERTを用いた方法なども有名である。特にBERTは膨大なコーパスをもとに作られた事前学習済みモデルでファインチューニングすれば精度の高いモデルの実現が期待できることが報告されている。

In [ ]:
#重みを初期化を考慮したモデル（参考）

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
#GPUで並列化処理による高速化。
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#LSTM分類器クラスの宣言
#ハイパパラメータは辞書で宣言
class LSTMClassifier(nn.Module):
    #各層の宣言　6層
    def __init__(self, **params):
        super(LSTMClassifier, self).__init__()
        
        #埋め込み層の次元の宣言
        self.word_embeddings = nn.Embedding(params["vocab_size"], params["embedding_dim"], padding_idx=0)
        #dropout
        self.dropout1 = nn.Dropout(params["dropout"])
        #lstmのインスタンス化
        self.lstm1 = nn.LSTM(params["embedding_dim"], params["hidden_dim"], batch_first=True)
        self.dropout2 = nn.Dropout(params["dropout"])
        self.linear = nn.Linear(params["hidden_dim"], params["tag_size"])
        self.softmax = nn.LogSoftmax(dim=1)

        # 重みを初期化
        nn.init.normal_(self.word_embeddings.weight, std=0.01)
        nn.init.normal_(self.lstm1.weight_ih_l0, std=1/math.sqrt(params["embedding_dim"]))
        nn.init.normal_(self.lstm1.weight_hh_l0, std=1/math.sqrt(params["hidden_dim"]))
        nn.init.zeros_(self.lstm1.bias_ih_l0)
        nn.init.zeros_(self.lstm1.bias_hh_l0)
        self.linear.weight = self.word_embeddings.weight  # 重み共有
        nn.init.zeros_(self.linear.bias)

    
    def forward(self, sentence, hidden1_prev, hidden2_prev):
        embeds = self.word_embeddings(sentence)
        embeds = self.dropout1(embeds)
        _, lstm1_out = self.lstm1(embeds, hidden1_prev)
        lstm1_out = self.dropout2(lstm1_out[0])
        out = self.linear(lstm1_out.view(-1, params["hidden_dim"]))
        tag_scores = self.softmax(out)
        return tag_scores